# NEU Evaluation & Benchmarking

Compares all models on:
- **Perplexity** — val loss on FineWeb
- **Hellaswag** — commonsense reasoning
- **ARC-Easy** — question answering
- **PIQA** — physical intuition
- **WinoGrande** — coreference resolution
- **Tokens/sec** — inference throughput

Produces the ablation table from the NEU-PROTOTYPE-PLAN.

In [ ]:
# Setup
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install transformers datasets tiktoken tqdm wandb
!pip install mamba-ssm flash-attn --no-build-isolation 2>/dev/null || true

import torch
import time
import math
from pathlib import Path

import numpy as np
from tqdm.auto import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# Load checkpoints from Google Drive
from google.colab import drive
drive.mount('/content/drive')
CKPT_DIR = Path('/content/drive/MyDrive/neu_checkpoints')

Looking in indexes: https://download.pytorch.org/whl/cu121
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 16.2 MB/s eta 0:00:0000:010:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.7/121.7 kB 2.8 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 74.4 MB/s eta 0:00:00:00:0100:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.6/65.6 kB 13.1 MB/s eta 0:00:00

In [ ]:
import random

import wandb

# Start a new wandb run to track this script.
run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="asdev",
    # Set the wandb project where this run will be logged.
    project="new",
    # Track hyperparameters and run metadata.
    config={
        "learning_rate": 0.02,
        "architecture": "CNN",
        "dataset": "CIFAR-100",
        "epochs": 10,
    },
)

# Simulate training.
epochs = 10
offset = random.random() / 5
for epoch in range(2, epochs):
    acc = 1 - 2**-epoch - random.random() / epoch - offset
    loss = 2**-epoch + random.random() / epoch + offset

    # Log metrics to wandb.
    run.log({"acc": acc, "loss": loss})

# Finish the run and upload any remaining data.
run.finish()

## Load All Model Checkpoints

In [ ]:
# Rebuild all model architectures (same code as training notebooks)
class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.eps = eps
    def forward(self, x):
        return torch.nn.functional.rms_norm(x, self.weight.shape, self.weight, self.eps)

_HAS_FLASH = False
try:
    from flash_attn import flash_attn_func
    _HAS_FLASH = True
except:
    pass

# (Re-define Transformer, Mamba2, FixedHybrid, NeuModel here — same as training notebooks)
# Skipping full re-definition for brevity; assume model classes are defined.

CHECKPOINTS = {
    'transformer':  CKPT_DIR / 'transformer-150M-final.pt',
    'mamba2':        CKPT_DIR / 'mamba2-150M-final.pt',
    'fixed_hybrid':  CKPT_DIR / 'fixed-hybrid-150M-final.pt',
    'neu_ps0':       CKPT_DIR / 'neu-768l12-ps0-final.pt',
    'neu_ps1':       CKPT_DIR / 'neu-768l12-ps1-final.pt',
}

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

def load_model(name, checkpoint_path):
    if not checkpoint_path.exists():
        print(f'WARNING: {checkpoint_path} not found, skipping {name}')
        return None
    ckpt = torch.load(checkpoint_path, map_location=DEVICE)
    state = ckpt['model']
    # ... build model and load_state_dict(state)
    print(f'Loaded {name}: {count_parameters(model):,} params')
    model.eval()
    return model

## Benchmarking Harness

In [ ]:
from transformers import pipeline

def evaluate_hellaswag(model, tokenizer, n=100, seed=42):
    """Zero-shot Hellaswag using pipeline."""
    import wandb
    
    # Use evaluate library for quick benchmark
    try:
        import evaluate
        metric = evaluate.load('hellaswag')
        # For full eval, use: metric.compute(predictions=..., references=...)
        # Simplified: use pipeline for quick check
    except:
        pass
    
    # Quick approximation using perplexity proxy
    return None  # Placeholder

def benchmark_perplexity(model, dataloader, device, nsamples=100):
    """Compute perplexity on a sample of validation data."""
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    
    with torch.no_grad():
        for i, batch in enumerate(dataloader):
            if i >= nsamples:
                break
            input_ids = batch['input_ids'].to(device)
            targets = input_ids.clone()
            
            result = model(input_ids, targets=targets)
            loss = result['loss'].item()
            
            # Count non-padded tokens
            mask = (input_ids != tokenizer.pad_token_id)
            n_tokens = mask.sum().item()
            
            total_loss += loss * n_tokens
            total_tokens += n_tokens
    
    avg_loss = total_loss / max(total_tokens, 1)
    perplexity = math.exp(avg_loss)
    return perplexity

def benchmark_throughput(model, seq_len=2048, batch_sizes=[1, 8, 32], device='cuda'):
    """Measure tokens/sec at different batch sizes."""
    model.eval()
    results = {}
    
    for bs in batch_sizes:
        input_ids = torch.randint(0, 32000, (bs, seq_len), device=device)
        targets = input_ids.clone()
        
        # Warmup
        with torch.no_grad():
            for _ in range(3):
                _ = model(input_ids)
        
        # Timed run
        torch.cuda.synchronize()
        t0 = time.time()
        n_runs = 20
        with torch.no_grad():
            for _ in range(n_runs):
                _ = model(input_ids)
        torch.cuda.synchronize()
        elapsed = time.time() - t0
        tokens_per_sec = (bs * seq_len * n_runs) / elapsed
        results[bs] = tokens_per_sec
        print(f'  batch={bs}: {tokens_per_sec:,.0f} tokens/sec')
    
    return results

def flops_per_token(model, seq_len=2048):
    """Estimate FLOPs/token for each model type.
    
    Approximate:
    - Transformer: ~2 * params * seq_len FLOPs
    - Mamba-2:     ~2 * params * seq_len * 0.7 FLOPs (no attention)
    - FixedHybrid: ~2 * params * seq_len * 0.85 FLOPs
    - NeuModel:   ~2 * params * seq_len * (0.4 cheap + 0.4 ssm + 0.2 attn)
                 = ~2 * params * seq_len * 0.65 (with ~40% on cheap)
    """
    params = sum(p.numel() for p in model.parameters())
    # This is a rough heuristic — actual FLOPs depend on hardware
    return 2 * params  # per token (simplified)

## Run Full Evaluation Table

In [ ]:
results = {}

for name, ckpt_path in CHECKPOINTS.items():
    if not ckpt_path.exists():
        print(f'Skipping {name} (not found)')
        continue
    
    print(f'\n{"="*60}')
    print(f'Evaluating: {name}')
    print(f'{"="*60}')
    
    model = load_model(name, ckpt_path)
    if model is None:
        continue
    
    # Perplexity
    print('Computing perplexity...')
    ppl = benchmark_perplexity(model, val_loader, DEVICE, nsamples=100)
    print(f'  Val PPL: {ppl:.4f}')
    
    # Throughput
    print('Benchmarking throughput...')
    throughput = benchmark_throughput(model, seq_len=2048, batch_sizes=[1, 8])
    
    results[name] = {
        'perplexity': ppl,
        'throughput_bs1': throughput.get(1, 0),
        'throughput_bs8': throughput.get(8, 0),
        'params': count_parameters(model),
    }

print('\n' + '='*60)
print('RESULTS SUMMARY')
print('='*60)
print(f'{"Model":<20} {"Params":>10} {"PPL":>8} {"Tok/s (B=1)":>14} {"Tok/s (B=8)":>14}')
print('-'*70)
for name, r in results.items():
    print(f'{name:<20} {r["params"]:>10,} {r["perplexity"]:>8.4f} '
          f'{r["throughput_bs1"]:>14,.0f} {r["throughput_bs8"]:>14,.0f}')

## Downstream Benchmarks (Hellaswag, ARC, PIQA)

In [ ]:
def run_lm_eval_harness(model_name, model, tokenizer, num_fewshot=0):
    """
    Run lm-evaluation-harness for standard benchmarks.
    
    pip install lm-evaluation-harness
    
    Usage:
    from lm_eval import evaluator
    results = evaluator.simple_evaluate(
        model=model,
        model_args=f'parallelize=True,padding=left',
        tasks=['hellaswag', 'arc_easy', 'piqa', 'winogrande'],
        num_fewshot=0,
        batch_size=8,
    )
    print(results.results)
    """
    !pip install lm-evaluation-harness 2>/dev/null
    print('lm-evaluation-harness installed. Run separately with:')
    print(f'  evaluator.simple_evaluate(model={model_name}, ...)')
    return None  # Manual step

# Example command for Colab:
# !python -m lm_eval --model hf \
#     --model_args pretrained=/path/to/checkpoint,parallelize=True \
#     --tasks hellaswag,arc_easy,piqa,winogrande \
#     --batch_size 8 \
#     --output_path /content/eval_results.json

## The Ablation Table

In [ ]:
# Build the canonical ablation table from NEU-PROTOTYPE-PLAN
# Fill in with actual benchmark numbers once runs complete

table = {
    'Config': [
        'Pure Transformer (baseline)',
        'Pure Mamba-2 (baseline)',
        'Fixed Hybrid Jamba-style (baseline)',
        'NeuModel (stateless routing)',
        'NeuModel (state-conditioned)',
    ],
    'Val Loss': ['TBD', 'TBD', 'TBD', 'TBD', 'TBD'],
    'Hellaswag': ['TBD', 'TBD', 'TBD', 'TBD', 'TBD'],
    'ARC-Easy': ['TBD', 'TBD', 'TBD', 'TBD', 'TBD'],
    'PIQA': ['TBD', 'TBD', 'TBD', 'TBD', 'TBD'],
    'Tok/sec (B=1)': ['TBD', 'TBD', 'TBD', 'TBD', 'TBD'],
    'FLOPs/token (est)': ['TBD', 'TBD', 'TBD', 'TBD', 'TBD'],
}

import pandas as pd
df = pd.DataFrame(table)
print('\nABLATION TABLE')
print('='*80)
print(df.to_string(index=False))
print('\nNote: Fill in TBD cells with actual benchmark results from Colab runs.')
print('\nKey question: Does NeuModel achieve equal/better quality at fewer FLOPs?')
print('Expected: NeuModel routes ~40% of tokens to cheap path,')
print('         giving ~35-40% FLOP reduction with similar quality.')

## Gate Distribution Analysis

In [ ]:
# Analyze gate distributions from NeuModel checkpoints
def analyze_gates(checkpoint_path, model, dataloader, device, nsamples=20):
    """Analyze routing decisions across layers and token types."""
    model.eval()
    
    all_gate_dist = []
    
    with torch.no_grad():
        for i, batch in enumerate(dataloader):
            if i >= nsamples:
                break
            input_ids = batch['input_ids'].to(device)
            targets = input_ids.clone()
            
            result = model(input_ids, targets=targets)
            gates = result['gates']  # list of (B, T, 3) per layer
            
            for layer_gates in gates:
                # Average over batch and sequence
                avg = layer_gates.mean(dim=(0, 1)).cpu().numpy()
                all_gate_dist.append(avg)
    
    import numpy as np
    all_gate_dist = np.array(all_gate_dist)  # (n_layers * samples, 3)
    
    # Per-layer averages
    n_layers = len(gates)
    n_samples = len(all_gate_dist) // n_layers
    per_layer = all_gate_dist.reshape(n_layers, n_samples, 3).mean(axis=1)
    
    return per_layer  # (n_layers, 3)

# Plot
import matplotlib.pyplot as plt

if 'neu_ps1' in results and (CKPT_DIR / 'neu-768l12-ps1-final.pt').exists():
    model = load_model('neu_ps1', CKPT_DIR / 'neu-768l12-ps1-final.pt')
    per_layer = analyze_gates(CKPT_DIR / 'neu-768l12-ps1-final.pt', model, val_loader, DEVICE)
    
    layers = range(len(per_layer))
    
    plt.figure(figsize=(10, 4))
    plt.plot(layers, per_layer[:, 0], label='Cheap', marker='o')
    plt.plot(layers, per_layer[:, 1], label='SSM', marker='s')
    plt.plot(layers, per_layer[:, 2], label='Attn', marker='^')
    plt.xlabel('Layer')
    plt.ylabel('Average Gate Value')
    plt.title('NeuModel: Per-Layer Gate Distribution')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.savefig('/content/drive/MyDrive/neu_checkpoints/gate_distribution.png', dpi=150)
    plt.show()
    print('Saved: gate_distribution.png')
else:
    print('NeuModel checkpoint not found — run training first')